In [2]:
import torch
from torch.utils.data import Dataset, DataLoader

class StudyDataset(Dataset):

    def __init__(self, x: torch.Rensor, y: torch.Tensor) -> None:
        self.x = x
        self.y = y

    def __len__(self) -> int:
        return len(self.x)
    
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.x[idx], self.y[idx]


x = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0]])
y = torch.tensor([[10.0], [20.0], [30.0], [40.0], [50.0], [60.0]])

dataset = StudyDataset(x, y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

print("데이터 개수:", len(dataset))
print("첫 번째 샘플:", dataset[0])
for x_batch, y_batch in loader:
    print("X batch:", x_batch.squeeze().tolist(), "| y batch:", y_batch.squeeze().tolist())

데이터 개수: 6
첫 번째 샘플: (tensor([1.]), tensor([10.]))
X batch: [5.0, 3.0] | y batch: [50.0, 30.0]
X batch: [2.0, 1.0] | y batch: [20.0, 10.0]
X batch: [4.0, 6.0] | y batch: [40.0, 60.0]


In [6]:
import torch
import torch.nn as nn


class RegressionModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layer1 = nn.Linear(1, 10)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(10, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.layer1(x)
        x = self.relu(x)
        return self.layer2(x)


model = RegressionModel()
sample_X = torch.tensor([[1.0], [2.0], [3.0]])
print(model)
print("예측 shape:", model(sample_X).shape)

RegressionModel(
  (layer1): Linear(in_features=1, out_features=10, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=10, out_features=1, bias=True)
)
예측 shape: torch.Size([3, 1])


In [10]:
import torch
import torch.nn as nn

x = torch.tensor(2.0, requires_grad=True)
y_square = x**2
y_square.backward()
print("y = x²에서 x=2의 gradient:", x.grad.item()) 

linear_model = nn.Linear(1, 1)
X_small = torch.tensor([[1.0], [2.0], [3.0]])
y_small = torch.tensor([[2.0], [4.0], [6.0]])
loss = nn.MSELoss()(linear_model(X_small), y_small)
loss.backward()
print("weight gradient:", linear_model.weight.grad)
print("bias gradient:", linear_model.bias.grad)

y = x²에서 x=2의 gradient: 4.0
weight gradient: tensor([[-20.4607]])
bias gradient: tensor([-8.5977])


In [12]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
x_train = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0]])
y_train = 2 * x_train
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=2, shuffle=True)

model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

epochs = 300
for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for x_batch, y_batch in train_loader:
        prediction = model(x_batch)
        loss = criterion(prediction, y_batch)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1:>3} |loss: {total_loss / len(train_loader.dataset):.6f}")

with torch.no_grad():
    print("x=10 예측:", model(torch.tensor([[10.0]])).item())

Epoch 100 |loss: 0.000000
Epoch 200 |loss: 0.000000
Epoch 300 |loss: 0.000000
x=10 예측: 8.475393295288086


In [18]:
import torch
from torch.utils.data import TensorDataset, random_split

full_dataset = TensorDataset(torch.randn(1000, 5), torch.randn(1000, 1))
generator = torch.Generator().manual_seed(42)  # 같은 분할을 재현한다.
train_set, val_set, test_set = random_split(full_dataset, [700, 150, 150], generator=generator)

print(f"Train: {len(train_set)}, Validation: {len(val_set)}, Test: {len(test_set)}")

# model과 criterion은 앞의 '전체 학습 루프' 셀에서 만들어진다.
# 셀을 순서와 다르게 실행한 경우에는 예외 대신 필요한 실행 순서를 안내한다.
if "model" not in globals() or "criterion" not in globals():
    print("평가를 건너뜁니다: 먼저 '4. 전체 학습 루프' 셀을 실행하세요.")
else:
    model.eval()
    X_test = torch.tensor([[10.0], [20.0], [30.0]])
    y_test = 2 * X_test
    with torch.no_grad():
        prediction = model(X_test)
    test_loss = criterion(prediction, y_test)
    print("Test MSE:", test_loss.item())


Train: 700, Validation: 150, Test: 150
Test MSE: 671.9749145507812


In [20]:
import torch 
import torch.nn as nn

class RegularizedModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

regularized_model = RegularizedModel()
print(regularized_model)
print("출력 shape:", regularized_model(torch.randn(4, 10)).shape)

RegularizedModel(
  (network): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)
출력 shape: torch.Size([4, 1])


In [25]:
import torch
import torch.nn as nn


class CNN(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(64 * 7 * 7, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)


cnn = CNN()
mnist_batch = torch.randn(8, 1, 28, 28)
print("입력:", mnist_batch.shape, "| 출력(logit):", cnn(mnist_batch).shape)

입력: torch.Size([8, 1, 28, 28]) | 출력(logit): torch.Size([8, 10])


In [ ]:
import torch 
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size: int = 1, hidden_size: int = 64, output_size: int = 1) -> None:
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        outputs, _ = self.lstm(x)
        last_output = outputs[:, -1, :]
        return self.fc(last_output)

lstm = LSTMModel()
time_series_batch = torch.randn(32, 10, 1)
print("입력:", time_series_batch.shape, "| 출력:", lstm(time_series_batch).shape)

입력: torch.Size([32, 10, 1]) | 출력: torch.Size([32, 1])


In [31]:
import torch
import torch.nn as nn

class TransformerClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int = 10_000,
        embedding_dim: int = 128,
        num_heads: int = 4,
        hidden_dim: int = 256,
        num_layers: int = 2,
        output_dim: int = 2,
        max_length: int = 128,
    ) -> None:
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(max_length, embedding_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embedding_dim, output_dim)\

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.token_embedding(x) + self.position_embedding(positions)
        x = self.transformer(x)
        return self.fc(x[:, 0, :])

transformer = TransformerClassifier()
token_batch = torch.randint(0, 10_000, (32, 20))
print("입력 Token ID:", token_batch.shape, "| 출력(logit):", transformer(token_batch).shape)

입력 Token ID: torch.Size([32, 20]) | 출력(logit): torch.Size([32, 2])
